In [1]:
import os
import glob
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

# 1. Define Absolute Paths
raw_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\reprojected"

os.makedirs(output_dir, exist_ok=True)

# 2. Define Target CRS (UTM Zone 43N for Bengaluru)
target_crs = 'EPSG:32643'

# 3. Gather all .tif files across all subfolders in the raw directory
tif_files = glob.glob(os.path.join(raw_dir, '**', '*.tif'), recursive=True)
print(f"Found {len(tif_files)} rasters to reproject. Starting...\n")

# 4. Processing Loop
for filepath in tif_files:
    filename = os.path.basename(filepath)
    parent_folder = os.path.basename(os.path.dirname(filepath))
    out_path = os.path.join(output_dir, filename)
    
    # Apply strict resampling rule based on data type
    if parent_folder == 'landcover':
        resamp_method = Resampling.nearest
    else:
        resamp_method = Resampling.bilinear

    # Open source raster
    with rasterio.open(filepath) as src:
        # Calculate the new spatial transform matrix and dimensions
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds)
        
        # Copy metadata and update with new CRS and dimensions
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': target_crs,
            'transform': transform,
            'width': width,
            'height': height
        })
        
        # Execute the reprojection and write to disk
        with rasterio.open(out_path, 'w', **kwargs) as dest:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dest, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=target_crs,
                resampling=resamp_method
            )
            
    print(f"Reprojected: {filename} ({parent_folder})")

print("\nCheckpoint: All layers converted to same CRS (EPSG:32643).")

Found 72 rasters to reproject. Starting...

Reprojected: pet_apr.tif (climate)
Reprojected: pet_aug.tif (climate)
Reprojected: pet_dec.tif (climate)
Reprojected: pet_feb.tif (climate)
Reprojected: pet_jan.tif (climate)
Reprojected: pet_jul.tif (climate)
Reprojected: pet_jun.tif (climate)
Reprojected: pet_mar.tif (climate)
Reprojected: pet_may.tif (climate)
Reprojected: pet_nov.tif (climate)
Reprojected: pet_oct.tif (climate)
Reprojected: pet_sep.tif (climate)
Reprojected: rain_apr.tif (climate)
Reprojected: rain_aug.tif (climate)
Reprojected: rain_dec.tif (climate)
Reprojected: rain_feb.tif (climate)
Reprojected: rain_jan.tif (climate)
Reprojected: rain_jul.tif (climate)
Reprojected: rain_jun.tif (climate)
Reprojected: rain_mar.tif (climate)
Reprojected: rain_may.tif (climate)
Reprojected: rain_nov.tif (climate)
Reprojected: rain_oct.tif (climate)
Reprojected: rain_sep.tif (climate)
Reprojected: temp_apr.tif (climate)
Reprojected: temp_aug.tif (climate)
Reprojected: temp_dec.tif (clima

In [3]:
import os
import glob
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine

# Paths
input_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\reprojected"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\resampled"

os.makedirs(output_dir, exist_ok=True)

# Target resolution (meters)
target_res = 250

# Collect rasters
tif_files = glob.glob(os.path.join(input_dir, '*.tif'))
print(f"Found {len(tif_files)} rasters. Resampling to {target_res}m...\n")

for filepath in tif_files:
    filename = os.path.basename(filepath)
    out_path = os.path.join(output_dir, filename)

    if 'worldcover' in filename.lower():
        resamp_method = Resampling.nearest
    else:
        resamp_method = Resampling.bilinear

    with rasterio.open(filepath) as src:

        # Current resolution
        xres, yres = src.res

        # Scaling factor
        scale_x = xres / target_res
        scale_y = abs(yres) / target_res

        # New dimensions
        new_width = int(src.width * scale_x)
        new_height = int(src.height * scale_y)

        # Safety check
        if new_width <= 0 or new_height <= 0:
            print(f"Skipping {filename}: invalid dimensions")
            continue

        # New transform
        new_transform = src.transform * Affine.scale(
            src.width / new_width,
            src.height / new_height
        )

        profile = src.profile.copy()
        profile.update({
            'height': new_height,
            'width': new_width,
            'transform': new_transform
        })

        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=resamp_method
        )

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data)

    print(f"Resampled: {filename}")

print("\nCheckpoint: Every raster resampled to 250m.")

Found 72 rasters. Resampling to 250m...

Resampled: aspect.tif
Resampled: B11.tif
Resampled: B12.tif
Resampled: B2.tif
Resampled: B3.tif
Resampled: B4.tif
Resampled: B8.tif
Resampled: bd_0-5.tif
Resampled: bd_15-30.tif
Resampled: bd_5-15.tif
Resampled: bsi.tif
Resampled: cec_0-5.tif
Resampled: cec_15-30.tif
Resampled: cec_5-15.tif
Resampled: clay_0-5.tif
Resampled: clay_15-30.tif
Resampled: clay_5-15.tif
Resampled: dem.tif
Resampled: evi.tif
Resampled: ndmi.tif
Resampled: ndvi.tif
Resampled: pet_apr.tif
Resampled: pet_aug.tif
Resampled: pet_dec.tif
Resampled: pet_feb.tif
Resampled: pet_jan.tif
Resampled: pet_jul.tif
Resampled: pet_jun.tif
Resampled: pet_mar.tif
Resampled: pet_may.tif
Resampled: pet_nov.tif
Resampled: pet_oct.tif
Resampled: pet_sep.tif
Resampled: ph_0-5.tif
Resampled: ph_15-30.tif
Resampled: ph_5-15.tif
Resampled: rain_apr.tif
Resampled: rain_aug.tif
Resampled: rain_dec.tif
Resampled: rain_feb.tif
Resampled: rain_jan.tif
Resampled: rain_jul.tif
Resampled: rain_jun.tif
R

In [6]:
import os
import glob
import rasterio
from rasterio.mask import mask
import geopandas as gpd

# 1. Define Absolute Paths
input_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\resampled"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clipped"
aoi_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\aoi\aoi.geojson"

os.makedirs(output_dir, exist_ok=True)

# 2. Load and Reproject the AOI
print("Loading and aligning boundary...")
aoi = gpd.read_file(aoi_path)
# Project the AOI to match the rasters (UTM Zone 43N)
aoi_proj = aoi.to_crs('EPSG:32643')
# Extract the geometry for the clipping function
geoms = aoi_proj.geometry.values

# 3. Gather resampled rasters
tif_files = glob.glob(os.path.join(input_dir, '*.tif'))
print(f"Found {len(tif_files)} rasters. Initiating spatial crop...\n")

# 4. Processing Loop
for filepath in tif_files:
    filename = os.path.basename(filepath)
    out_path = os.path.join(output_dir, filename)
    
    try:
        with rasterio.open(filepath) as src:
            # crop=True forces the matrix to shrink exactly to the boundary's bounding box
            out_image, out_transform = mask(src, geoms, crop=True)
            out_meta = src.meta.copy()
            
            # Update metadata with new, smaller dimensions
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform
            })
            
            # Write the final clipped raster
            with rasterio.open(out_path, "w", **out_meta) as dest:
                dest.write(out_image)
                
        print(f"Clipped: {filename}")
        
    except ValueError as e:
        print(f"Failed to clip {filename}: {e}")

print("\nCheckpoint: No extra area. All layers perfectly aligned, resampled, and clipped.")

Loading and aligning boundary...
Found 60 rasters. Initiating spatial crop...

Clipped: aspect.tif
Clipped: B11.tif
Clipped: B12.tif
Clipped: B2.tif
Clipped: B3.tif
Clipped: B4.tif
Clipped: B8.tif
Clipped: bd_0-5.tif
Clipped: bd_15-30.tif
Clipped: bd_5-15.tif
Clipped: bsi.tif
Clipped: cec_0-5.tif
Clipped: cec_15-30.tif
Clipped: cec_5-15.tif
Clipped: clay_0-5.tif
Clipped: clay_15-30.tif
Clipped: clay_5-15.tif
Clipped: dem.tif
Clipped: evi.tif
Clipped: ndmi.tif
Clipped: ndvi.tif
Clipped: pet_apr.tif
Clipped: pet_aug.tif
Clipped: pet_dec.tif
Clipped: pet_feb.tif
Clipped: pet_jan.tif
Clipped: pet_jul.tif
Clipped: pet_jun.tif
Clipped: pet_mar.tif
Clipped: pet_may.tif
Clipped: pet_nov.tif
Clipped: pet_oct.tif
Clipped: pet_sep.tif
Clipped: ph_0-5.tif
Clipped: ph_15-30.tif
Clipped: ph_5-15.tif
Clipped: rain_apr.tif
Clipped: rain_aug.tif
Clipped: rain_dec.tif
Clipped: rain_feb.tif
Clipped: rain_jan.tif
Clipped: rain_jul.tif
Clipped: rain_jun.tif
Clipped: rain_mar.tif
Clipped: rain_may.tif
Clipp

In [7]:
import os
import glob
import numpy as np
import rasterio
from rasterio.fill import fillnodata

# 1. Define Absolute Paths
input_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clipped"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\filled"

os.makedirs(output_dir, exist_ok=True)

# 2. Gather clipped rasters
tif_files = glob.glob(os.path.join(input_dir, '*.tif'))
print(f"Checking {len(tif_files)} rasters for missing values...\n")

# 3. Processing Loop
for filepath in tif_files:
    filename = os.path.basename(filepath)
    out_path = os.path.join(output_dir, filename)
    
    with rasterio.open(filepath) as src:
        profile = src.profile
        arr = src.read(1)
        nodata_val = src.nodata

        # Create a boolean mask where True = Valid Data, False = Missing/NoData
        if nodata_val is not None:
            valid_mask = (arr != nodata_val)
            # Catch stray NaNs just in case
            valid_mask = valid_mask & ~np.isnan(arr) 
        else:
            valid_mask = ~np.isnan(arr)
            nodata_val = np.nan

        # If the array has missing values (but isn't completely empty)
        if not valid_mask.all() and valid_mask.any():
            print(f"Interpolating missing values in: {filename}")
            
            # fillnodata uses surrounding valid pixels to calculate the missing values
            # max_search_distance limits how far it looks, preventing it from filling the exterior boundary
            filled_arr = fillnodata(arr, mask=valid_mask, max_search_distance=5, smoothing_iterations=0)
        else:
            filled_arr = arr
            
        # Write the cleaned array to the new directory
        with rasterio.open(out_path, 'w', **profile) as dest:
            dest.write(filled_arr, 1)

print("\nCheckpoint: No missing core variables.")


Checking 60 rasters for missing values...

Interpolating missing values in: aspect.tif
Interpolating missing values in: pet_apr.tif
Interpolating missing values in: pet_aug.tif
Interpolating missing values in: pet_dec.tif
Interpolating missing values in: pet_feb.tif
Interpolating missing values in: pet_jan.tif
Interpolating missing values in: pet_jul.tif
Interpolating missing values in: pet_jun.tif
Interpolating missing values in: pet_mar.tif
Interpolating missing values in: pet_may.tif
Interpolating missing values in: pet_nov.tif
Interpolating missing values in: pet_oct.tif
Interpolating missing values in: pet_sep.tif
Interpolating missing values in: rain_apr.tif
Interpolating missing values in: rain_aug.tif
Interpolating missing values in: rain_dec.tif
Interpolating missing values in: rain_feb.tif
Interpolating missing values in: rain_jan.tif
Interpolating missing values in: rain_jul.tif
Interpolating missing values in: rain_jun.tif
Interpolating missing values in: rain_mar.tif
Inter

In [8]:
import os
import glob
import numpy as np
import rasterio

# 1. Define Absolute Paths
input_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\filled"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\outliers_removed"

os.makedirs(output_dir, exist_ok=True)

# 2. Define the target variables to check for outliers
# We use partial string matching to catch all variations of temp, soc, ph, etc.
target_keywords = ['soc', 'ph', 'bulk', 'bdfiod', 'clay', 'temp']

# 3. Gather filled rasters
tif_files = glob.glob(os.path.join(input_dir, '*.tif'))
print(f"Scanning {len(tif_files)} rasters for outliers...\n")

# 4. Processing Loop
for filepath in tif_files:
    filename = os.path.basename(filepath)
    out_path = os.path.join(output_dir, filename)
    
    # Check if the file is one of the target variables
    is_target = any(keyword in filename.lower() for keyword in target_keywords)
    
    with rasterio.open(filepath) as src:
        profile = src.profile
        arr = src.read(1).astype('float32') # Ensure float for NaN assignment
        nodata_val = src.nodata
        
        # Standardize NoData handling
        if nodata_val is None:
            nodata_val = -9999.0
            profile.update(nodata=nodata_val)
            
        valid_mask = (arr != nodata_val) & ~np.isnan(arr)
        
        if is_target and valid_mask.any():
            # Isolate valid data to calculate clean statistics
            valid_data = arr[valid_mask]
            mean = np.mean(valid_data)
            std = np.std(valid_data)
            
            if std > 0:
                # Calculate Z-score
                z_scores = np.zeros_like(arr)
                z_scores[valid_mask] = (arr[valid_mask] - mean) / std
                
                # Flag extreme noise (|Z| > 3)
                outliers = (np.abs(z_scores) > 3) & valid_mask
                num_outliers = np.sum(outliers)
                
                # Remove outliers by setting them to NoData
                if num_outliers > 0:
                    arr[outliers] = nodata_val
                    print(f"Flagged and removed {num_outliers} outliers in: {filename}")
                else:
                    print(f"No extreme outliers found in: {filename}")
            else:
                print(f"Skipped {filename}: Variance is zero.")
        else:
            # For non-target files (or empty files), just pass them through
            if not is_target:
                pass # Silent pass for structural files like landcover/DEM
                
        # Write the cleaned array
        with rasterio.open(out_path, 'w', **profile) as dest:
            dest.write(arr, 1)

print("\nCheckpoint: Outliers flagged and extreme noise removed.")

Scanning 60 rasters for outliers...

No extreme outliers found in: clay_0-5.tif
No extreme outliers found in: clay_15-30.tif
No extreme outliers found in: clay_5-15.tif
No extreme outliers found in: ph_0-5.tif
No extreme outliers found in: ph_15-30.tif
No extreme outliers found in: ph_5-15.tif
No extreme outliers found in: soc_0-5.tif
Flagged and removed 2 outliers in: soc_15-30.tif
Flagged and removed 1 outliers in: soc_5-15.tif

Checkpoint: Outliers flagged and extreme noise removed.


In [9]:
import os
import glob
import numpy as np
import rasterio

# 1. Define Absolute Paths
input_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\outliers_removed"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\standardized"

os.makedirs(output_dir, exist_ok=True)

# Mapping months to days for potential precipitation conversion
days_in_month = {
    'jan': 31, 'feb': 28, 'mar': 31, 'apr': 30, 'may': 31, 'jun': 30,
    'jul': 31, 'aug': 31, 'sep': 30, 'oct': 31, 'nov': 30, 'dec': 31
}

# 2. Gather rasters
tif_files = glob.glob(os.path.join(input_dir, '*.tif'))
print(f"Standardizing units across {len(tif_files)} rasters...\n")

# 3. Processing Loop
for filepath in tif_files:
    filename = os.path.basename(filepath)
    out_path = os.path.join(output_dir, filename)
    
    with rasterio.open(filepath) as src:
        profile = src.profile
        arr = src.read(1).astype('float32')
        nodata_val = profile.get('nodata', -9999.0)
        
        valid_mask = (arr != nodata_val) & ~np.isnan(arr)
        
        if valid_mask.any():
            # Conversion 1: SOC (g/kg to %)
            if 'soc' in filename.lower() or 'orgc' in filename.lower():
                arr[valid_mask] = arr[valid_mask] / 10.0
                print(f"Converted SOC to %: {filename}")
                
            # Conversion 2: Temperature (Kelvin to Celsius)
            elif 'temp' in filename.lower():
                # Safety check: If average temp is > 100, it is in Kelvin.
                if np.mean(arr[valid_mask]) > 100:
                    arr[valid_mask] = arr[valid_mask] - 273.15
                    print(f"Converted Temp to Celsius: {filename}")
                else:
                    print(f"Temp already in Celsius, skipped: {filename}")
                    
            # Conversion 3: Rainfall (mm/day to mm/month)
            elif 'rain' in filename.lower() or 'prec' in filename.lower():
                # Safety check: If max rainfall is suspiciously low (< 25mm for a month), 
                # it is likely a daily average rate.
                if np.max(arr[valid_mask]) < 25:
                    month_str = filename.split('_')[1].split('.')[0].lower()
                    days = days_in_month.get(month_str, 30)
                    arr[valid_mask] = arr[valid_mask] * days
                    print(f"Converted Rain to mm/month ({days} days): {filename}")
                else:
                    print(f"Rain already in mm/month, skipped: {filename}")
                    
        # Write the standardized array to disk
        with rasterio.open(out_path, 'w', **profile) as dest:
            dest.write(arr, 1)

print("\nCheckpoint: Units consistent.")

Standardizing units across 60 rasters...

Rain already in mm/month, skipped: rain_apr.tif
Rain already in mm/month, skipped: rain_aug.tif
Rain already in mm/month, skipped: rain_dec.tif
Converted Rain to mm/month (28 days): rain_feb.tif
Converted Rain to mm/month (31 days): rain_jan.tif
Rain already in mm/month, skipped: rain_jul.tif
Rain already in mm/month, skipped: rain_jun.tif
Rain already in mm/month, skipped: rain_mar.tif
Rain already in mm/month, skipped: rain_may.tif
Rain already in mm/month, skipped: rain_nov.tif
Rain already in mm/month, skipped: rain_oct.tif
Rain already in mm/month, skipped: rain_sep.tif
Converted SOC to %: soc_0-5.tif
Converted SOC to %: soc_15-30.tif
Converted SOC to %: soc_5-15.tif

Checkpoint: Units consistent.


In [10]:
import os
import numpy as np
import rasterio

# 1. Define Absolute Paths
# Assuming the file passed through standardizing unchanged, we read from there.
input_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\standardized\worldcover.tif"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\encoded"

os.makedirs(output_dir, exist_ok=True)
out_path = os.path.join(output_dir, 'worldcover_encoded.tif')

# 2. Define the exact label encoding dictionary
# Format: {ESA_Code: New_Label}
encoding_map = {
    10: 1,  # Trees/Forest
    30: 2,  # Grassland
    40: 3,  # Cropland
    50: 4,  # Urban/Built-up
    90: 5   # Herbaceous Wetland
}

# 3. Process the Raster
if os.path.exists(input_path):
    with rasterio.open(input_path) as src:
        profile = src.profile
        arr = src.read(1)
        
        # Create an empty array of zeros (defaulting unmapped classes to 0)
        encoded_arr = np.zeros_like(arr, dtype=np.uint8)
        
        # Apply the mapping
        for esa_code, new_label in encoding_map.items():
            # Wherever the original array equals the ESA code, replace with new label
            encoded_arr[arr == esa_code] = new_label
            
        # Update profile to reflect standard 8-bit unsigned integer classes
        profile.update(dtype=rasterio.uint8, nodata=0)
        
        # Write to disk
        with rasterio.open(out_path, 'w', **profile) as dest:
            dest.write(encoded_arr, 1)
            
    print(f"Success: Landcover categories safely label-encoded.")
else:
    print(f"Error: Could not find {input_path}")

print("\nCheckpoint: Machine-readable.")

Success: Landcover categories safely label-encoded.

Checkpoint: Machine-readable.


In [13]:
import os
import glob
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

input_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\resampled"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\aligned"

os.makedirs(output_dir, exist_ok=True)

tif_files = glob.glob(os.path.join(input_dir, "*.tif"))

# Reference raster
reference_path = os.path.join(input_dir, "B2.tif")

with rasterio.open(reference_path) as ref:
    ref_meta = ref.meta.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width = ref.width
    ref_height = ref.height

for filepath in tif_files:
    filename = os.path.basename(filepath)
    out_path = os.path.join(output_dir, filename)

    with rasterio.open(filepath) as src:

        if 'worldcover' in filename.lower():
            resampling_method = Resampling.nearest
        else:
            resampling_method = Resampling.bilinear

        destination = np.empty(
            (src.count, ref_height, ref_width),
            dtype=src.dtypes[0]
        )

        for band in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, band),
                destination=destination[band - 1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=resampling_method
            )

        profile = ref_meta.copy()
        profile.update(
            count=src.count,
            dtype=src.dtypes[0]
        )

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(destination)

    print(f"Aligned: {filename}")

print("All rasters now share identical grid.")

Aligned: aspect.tif
Aligned: B11.tif
Aligned: B12.tif
Aligned: B2.tif
Aligned: B3.tif
Aligned: B4.tif
Aligned: B8.tif
Aligned: bd_0-5.tif
Aligned: bd_15-30.tif
Aligned: bd_5-15.tif
Aligned: bsi.tif
Aligned: cec_0-5.tif
Aligned: cec_15-30.tif
Aligned: cec_5-15.tif
Aligned: clay_0-5.tif
Aligned: clay_15-30.tif
Aligned: clay_5-15.tif
Aligned: dem.tif
Aligned: evi.tif
Aligned: ndmi.tif
Aligned: ndvi.tif
Aligned: pet_apr.tif
Aligned: pet_aug.tif
Aligned: pet_dec.tif
Aligned: pet_feb.tif
Aligned: pet_jan.tif
Aligned: pet_jul.tif
Aligned: pet_jun.tif
Aligned: pet_mar.tif
Aligned: pet_may.tif
Aligned: pet_nov.tif
Aligned: pet_oct.tif
Aligned: pet_sep.tif
Aligned: ph_0-5.tif
Aligned: ph_15-30.tif
Aligned: ph_5-15.tif
Aligned: rain_apr.tif
Aligned: rain_aug.tif
Aligned: rain_dec.tif
Aligned: rain_feb.tif
Aligned: rain_jan.tif
Aligned: rain_jul.tif
Aligned: rain_jun.tif
Aligned: rain_mar.tif
Aligned: rain_may.tif
Aligned: rain_nov.tif
Aligned: rain_oct.tif
Aligned: rain_sep.tif
Aligned: sand_0-5.

In [14]:
for f in glob.glob(os.path.join(output_dir, "*.tif")):
    with rasterio.open(f) as src:
        print(os.path.basename(f), src.shape, src.transform)

aspect.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
B11.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
B12.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
B2.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
B3.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
B4.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
B8.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
bd_0-5.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
bd_15-30.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
bd_5-15.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.70|
| 0.00, 0.00, 1.00|
bsi.tif (13, 11) | 250.14, 0.00, 768816.54|
| 0.00,-257.93, 1457847.7

In [16]:
import os
import glob
import numpy as np
import pandas as pd
import rasterio

aligned_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\aligned"
output_csv = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\master_scaled.csv"

all_files = glob.glob(os.path.join(aligned_dir, '*.tif'))

print(f"Flattening {len(all_files)} aligned spatial layers...")

data_dict = {}
valid_mask = None
reference_shape = None

for filepath in all_files:
    filename = os.path.basename(filepath)
    col_name = filename.replace('.tif', '')

    with rasterio.open(filepath) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata

        # First raster defines reference grid
        if reference_shape is None:
            reference_shape = arr.shape

        # Hard validation
        if arr.shape != reference_shape:
            raise ValueError(
                f"{filename} shape mismatch: {arr.shape} != {reference_shape}"
            )

        # Create one common valid mask
        if valid_mask is None:
            if nodata is not None:
                valid_mask = (arr != nodata) & (~np.isnan(arr))
            else:
                valid_mask = ~np.isnan(arr)

        data_dict[col_name] = arr[valid_mask]

df = pd.DataFrame(data_dict)

print(f"Extracted {len(df)} grid cells across {len(df.columns)} features.")

# scale continuous variables
categorical_cols = ['worldcover_encoded']
continuous_cols = [c for c in df.columns if c not in categorical_cols]

for col in continuous_cols:
    mean = df[col].mean()
    std = df[col].std()

    if std != 0:
        df[col] = (df[col] - mean) / std
    else:
        df[col] = 0.0

df.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")

Flattening 60 aligned spatial layers...
Extracted 143 grid cells across 60 features.
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\master_scaled.csv


In [17]:
# 4.5 Save the unscaled, clean tabular data first
clean_csv = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master.csv"
df.to_csv(clean_csv, index=False)
print(f"Saved unscaled baseline to: {clean_csv}")

Saved unscaled baseline to: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master.csv


In [20]:
import os
import numpy as np
import pandas as pd

# 1. Define Paths
clean_csv_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master.csv"
output_clean = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
output_scaled = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\master_scaled_engineered.csv"

# 2. Load the unscaled baseline data
print("Loading baseline data...")
print("Loading baseline data...")
df = pd.read_csv(clean_csv_path)

# Helper function to find exact column names ignoring case
def get_col(keyword):
    for col in df.columns:
        if keyword.lower() in col.lower():
            return col
    raise ValueError(f"Could not find column matching: {keyword}")

# Helper function to calculate depth-weighted average for 0-30cm profile
def get_weighted_0_30(var_name):
    col_0_5 = get_col(f'{var_name}_0-5')
    col_5_15 = get_col(f'{var_name}_5-15')
    col_15_30 = get_col(f'{var_name}_15-30')
    
    # Apply standard depth weights: 5cm, 10cm, 15cm (Total = 30cm)
    weighted_series = ((df[col_0_5] * 5) + (df[col_5_15] * 10) + (df[col_15_30] * 15)) / 30.0
    return weighted_series

print("Calculating 0-30cm weighted profiles...")
# Calculate the unified 0-30cm variables
df['clay_0_30'] = get_weighted_0_30('clay')
df['sand_0_30'] = get_weighted_0_30('sand')
df['silt_0_30'] = get_weighted_0_30('silt')
df['soc_0_30'] = get_weighted_0_30('soc') # Change to 'orgc' if your file uses that
df['bd_0_30'] = get_weighted_0_30('bd')

print("Calculating Texture Ratios...")
# 3. Texture Ratios (with safe division to prevent inf/NaN)
# 3. Texture Ratios (using the unified 0-30cm variables)
df['clay_sand_ratio'] = df['clay_0_30'] / df['sand_0_30'].replace(0, 1e-6)
df['silt_clay_ratio'] = df['silt_0_30'] / df['clay_0_30'].replace(0, 1e-6)
df['sand_silt_ratio'] = df['sand_0_30'] / df['silt_0_30'].replace(0, 1e-6)

print("Calculating SOM...")
# 4. Soil Organic Matter Estimate
# SOM = SOC * 1.724
df['som'] = df['soc_0_30'] * 1.724

print("Calculating Porosity...")
# 5. Soil Porosity
# Porosity = 1 - (BulkDensity / ParticleDensity)
# Assuming Bulk Density is in g/cm3. If it was in cg/cm3, divide by 100 first.
particle_density = 2.65
df['porosity'] = 1 - (df['bd_0_30'] / particle_density)
df['porosity'] = df['porosity'].clip(lower=0.01, upper=0.99)

# 6. Save the new Engineered Clean Dataset
df.to_csv(output_clean, index=False)
print(f"Engineered clean dataset saved to: {output_clean}")

# 7. Re-apply Standard Scaling for the ML-ready dataset
print("Re-scaling dataset with new features...")
categorical_cols = ['worldcover_encoded']
continuous_cols = [col for col in df.columns if col not in categorical_cols]

df_scaled = df.copy()

for col in continuous_cols:
    mean = df_scaled[col].mean()
    std = df_scaled[col].std()
    if std > 0:
        df_scaled[col] = (df_scaled[col] - mean) / std
    else:
        df_scaled[col] = 0.0

df_scaled.to_csv(output_scaled, index=False)
print(f"Engineered scaled dataset saved to: {output_scaled}")
print("\nCheckpoint: Step 3.1 Soil Feature Engineering complete.")

Loading baseline data...
Loading baseline data...
Calculating 0-30cm weighted profiles...
Calculating Texture Ratios...
Calculating SOM...
Calculating Porosity...
Engineered clean dataset saved to: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv
Re-scaling dataset with new features...
Engineered scaled dataset saved to: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\master_scaled_engineered.csv

Checkpoint: Step 3.1 Soil Feature Engineering complete.


In [21]:
import pandas as pd
import numpy as np

# 1. Load the engineered clean dataset (from Step 3.1)
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

# 2. Identify the raw bands from your dataset
# These names should match your band files (e.g., 'B8' for NIR, 'B11' for SWIR1)
nir = df['B8']
red = df['B4']
swir1 = df['B11']
blue = df['B2']

print("Calculating Advanced Remote Sensing Features...")

# 3. NDWI (Normalized Difference Water Index)
# Identifies moisture content - highly correlated with carbon mineralization
# NDWI = (NIR - SWIR1) / (NIR + SWIR1)
df['ndwi'] = (nir - swir1) / (nir + swir1).replace(0, 1e-6)

# 4. GNDVI (Green Normalized Difference Vegetation Index)
# Uses the green band instead of red. More sensitive to chlorophyll variation.
green = df['B3']
df['gndvi'] = (nir - green) / (nir + green).replace(0, 1e-6)

# 5. BI (Brightness Index)
# Identifies soil surface reflectance and roughness
# BI = sqrt((Red^2 + Green^2) / 2)
df['brightness_index'] = np.sqrt((red**2 + green**2) / 2)

# 6. NDTI (Normalized Difference Tillage Index)
# Proxy for crop residue and dry carbon on the soil surface
# NDTI = (SWIR1 - SWIR2) / (SWIR1 + SWIR2)
swir2 = df['B12']
df['ndti'] = (swir1 - swir2) / (swir1 + swir2).replace(0, 1e-6)

# 7. Save the updated clean dataset
df.to_csv(file_path, index=False)
print(f"Dataset updated with advanced RS features at: {file_path}")

# 8. Update the scaled version for the ML pipeline
output_scaled = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\master_scaled_engineered.csv"
categorical_cols = ['worldcover_encoded']
continuous_cols = [col for col in df.columns if col not in categorical_cols]

df_scaled = df.copy()
for col in continuous_cols:
    mean, std = df_scaled[col].mean(), df_scaled[col].std()
    df_scaled[col] = (df_scaled[col] - mean) / std if std > 0 else 0.0

df_scaled.to_csv(output_scaled, index=False)
print("Checkpoint: Advanced Remote Sensing features ready for model input.")

Calculating Advanced Remote Sensing Features...
Dataset updated with advanced RS features at: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv
Checkpoint: Advanced Remote Sensing features ready for model input.


In [62]:
# 3. Export to local
import requests
import os
output_dir_temporal = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\temporal"
os.makedirs(output_dir_temporal, exist_ok=True)

for band in stats_image.bandNames().getInfo():
    print(f"Downloading {band}...")
    out_path = os.path.join(output_dir_temporal, f"{band}.tif")
    
    url = stats_image.select(band).getDownloadURL({
        'scale': 10,
        'crs': 'EPSG:32643', # UTM 43N directly
        'region': aoi_ee,
        'format': 'GEO_TIFF'
    })
    
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(out_path, 'wb') as f:
            f.write(response.content)
        print(f"Saved: {out_path}")F
    else:
        print(f"Failed for {band}: {response.text}")            
        
print("\nCheckpoint: Temporal vegetation stats collected and UTM-projected.")

Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\temporal\ndvi_mean.tif
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\temporal\ndvi_std.tif
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\temporal\ndvi_max.tif
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\temporal\ndvi_min.tif
Saved: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\temporal\ndvi_amplitude.tif

Checkpoint: Temporal vegetation stats collected and UTM-projected.


In [24]:
import pandas as pd
import numpy as np

# 1. Load the latest dataset
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

# Ensure we use our unified 0-30cm SOC
soc_col = 'soc_0_30'

print("Initializing RothC Carbon Pools per pixel...")

# 2. Calculate IOM (Inert Organic Matter) using the Falloon Equation
# This is the "permanent" carbon that doesn't decay.
df['pool_IOM'] = 0.049 * (df[soc_col]**1.139)

# 3. Calculate Remaining SOC (Active Carbon)
active_soc = df[soc_col] - df['pool_IOM']

# 4. Standard Distribution for Grasslands (Spin-up defaults)
# In an equilibrium state for grasslands, the active carbon is typically 
# distributed roughly as follows (these can be refined with spin-up runs):
df['pool_DPM'] = active_soc * 0.01  # Very small, decays fast
df['pool_RPM'] = active_soc * 0.15  # Structural material
df['pool_BIO'] = active_soc * 0.03  # Microbial living mass
df['pool_HUM'] = active_soc * 0.81  # Stable humified carbon

# 5. Verification
# Check that the sum of pools equals the total SOC
check_sum = df[['pool_DPM', 'pool_RPM', 'pool_BIO', 'pool_HUM', 'pool_IOM']].sum(axis=1)
if np.allclose(check_sum, df[soc_col], atol=1e-3):
    print("Verification Successful: Carbon pools perfectly balanced.")

# 6. Save
df.to_csv(file_path, index=False)
print(f"Carbon pools initialized and saved to: {file_path}")

Initializing RothC Carbon Pools per pixel...
Carbon pools initialized and saved to: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv


In [27]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv")

# 1. Aggregate Climate Data
rain_cols = [c for c in df.columns if 'rain_' in c]
pet_cols = [c for c in df.columns if 'pet_' in c]

df['annual_rain'] = df[rain_cols].sum(axis=1)
df['annual_pet'] = df[pet_cols].sum(axis=1)
df['aridity_index'] = df['annual_rain'] / df['annual_pet'].replace(0, 1e-6)

# 2. Add CEC 0-30 (The one soil variable we didn't weight yet)
df['cec_0_30'] = ((df['cec_0-5'] * 5) + (df['cec_5-15'] * 10) + (df['cec_15-30'] * 15)) / 30.0

# 3. Identify and Drop Redundant Interval Columns
redundant_cols = [c for c in df.columns if any(suffix in c for suffix in ['0-5', '5-15', '15-30'])]
df_final = df.drop(columns=redundant_cols)

# 4. Save the "Final" ML-Ready Dataset
final_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\final_ml_ready.csv"
df_final.to_csv(final_path, index=False)

print(f"Cleaned! Redundant columns removed: {len(redundant_cols)}")
print(f"New Features: annual_rain, annual_pet, aridity_index, cec_0_30")
print(f"Final Column Count: {len(df_final.columns)}")

Cleaned! Redundant columns removed: 21
New Features: annual_rain, annual_pet, aridity_index, cec_0_30
Final Column Count: 62


In [26]:
import pandas as pd

# Load the current engineered file
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

# 1. Identify interval columns to drop (keeping only the 0_30 unified versions)
cols_to_drop = [c for c in df.columns if any(suffix in c for suffix in ['0-5', '5-15', '15-30'])]

# 2. Add Annual Climate Aggregates (Massive for ML performance)
rain_cols = [c for c in df.columns if 'rain_' in c]
pet_cols = [c for c in df.columns if 'pet_' in c]
df['annual_rain'] = df[rain_cols].sum(axis=1)
df['annual_pet'] = df[pet_cols].sum(axis=1)
df['aridity_index'] = df['annual_rain'] / df['annual_pet'].replace(0, 1e-6)

# 3. Final cleanup
df_final = df.drop(columns=cols_to_drop)

# Save the master file that goes into the model
final_ml_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\final_ml_ready.csv"
df_final.to_csv(final_ml_path, index=False)

print(f"Dropped {len(cols_to_drop)} redundant layers.")
print(f"Final feature count: {len(df_final.columns)}")

Dropped 21 redundant layers.
Final feature count: 61


In [28]:
import pandas as pd
import numpy as np

# 1. Load the dataset
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

# 2. Identify monthly column groups
# Assumes naming like 'rain_jan', 'temp_jan', 'pet_jan'
rain_cols = [c for c in df.columns if 'rain_' in c.lower()]
temp_cols = [c for c in df.columns if 'temp_' in c.lower()]
pet_cols = [c for c in df.columns if 'pet_' in c.lower()]

print(f"Processing climate features for {len(df)} pixels...")

# 3. Annual Totals & Means
# Rain_annual = Sum of monthly rain
df['annual_rain'] = df[rain_cols].sum(axis=1)

# Mean_temp = Average of monthly temperatures
df['mean_temp'] = df[temp_cols].mean(axis=1)

# 4. Variability & Seasonality
# Temp_std = Standard Deviation of monthly temperature
df['temp_std'] = df[temp_cols].std(axis=1)

# Rain_cv = Coefficient of Variation (Std / Mean)
# High CV = extreme seasonal monsoon/drought cycles
rain_mean = df[rain_cols].mean(axis=1).replace(0, 1e-6)
df['rain_cv'] = df[rain_cols].std(axis=1) / rain_mean

# 5. Aridity Index (Hydrologic Balance)
# AI = Precipitation / Potential Evapotranspiration
annual_pet = df[pet_cols].sum(axis=1).replace(0, 1e-6)
df['aridity_index'] = df['annual_rain'] / annual_pet

# 6. Save Updated Dataset
df.to_csv(file_path, index=False)

print("---")
print(f"Output generated: annual_rain, mean_temp, temp_std, rain_cv, aridity_index")
print("Checkpoint: Climate features consistent and verified.")

Processing climate features for 143 pixels...
---
Output generated: annual_rain, mean_temp, temp_std, rain_cv, aridity_index
Checkpoint: Climate features consistent and verified.


In [29]:
import pandas as pd
import numpy as np

# 1. Load the dataset
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

print("Engineering Terrain Features...")

# 2. Fix Aspect Circularity
# aspect_sin represents East-West exposure
# aspect_cos represents North-South exposure
df['aspect_sin'] = np.sin(np.radians(df['aspect']))
df['aspect_cos'] = np.cos(np.radians(df['aspect']))

# 3. Calculate TWI (Topographic Wetness Index)
# Formula: ln(alpha / tan(beta))
# alpha = Flow Accumulation (area contributing to a point)
# beta = Slope in radians
slope_rad = np.radians(df['slope']).replace(0, 1e-6) # Prevent division by zero

# If flow_acc isn't in your CSV yet, we use a proxy based on local curvature/slope 
# or ensure it was extracted from the DEM earlier.
if 'flow_acc' in df.columns:
    df['twi'] = np.log(df['flow_acc'] / np.tan(slope_rad))
else:
    # Proxy: Higher TWI in low-slope, high-curvature areas
    print("Warning: flow_acc not found. Using curvature-slope proxy for TWI.")
    df['twi'] = np.log(1 / np.tan(slope_rad))

# 4. Clean up
# Aspect itself is now redundant due to Sin/Cos
df = df.drop(columns=['aspect'])

# 5. Save Updated Dataset
df.to_csv(file_path, index=False)

print("---")
print("Output generated: aspect_sin, aspect_cos, twi")
print("Checkpoint: Terrain features consistent.")

Engineering Terrain Features...
---
Output generated: aspect_sin, aspect_cos, twi
Checkpoint: Terrain features consistent.


C:\Users\AYUSH SINGH\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [32]:
import pandas as pd

# 1. Load the engineered dataset
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

print("One-Hot Encoding Landcover Features...")

# 2. Define the Mapping based on Step 2.8
# 1=Forest, 2=Grassland, 3=Cropland, 4=Urban, 5=Wetland
lc_mapping = {
    1: 'lc_forest',
    2: 'lc_grassland',
    3: 'lc_crop',
    4: 'lc_urban',
    5: 'lc_wetland'
}

# 3. Create dummy variables
for code, name in lc_mapping.items():
    # Create a 1 if the code matches, otherwise 0
    df[name] = (df['worldcover'] == code).astype(int)

# 4. Cleanup
# We keep the 'worldcover_encoded' for reference, but the dummies 
# will do the heavy lifting in the ML model.
    
# 5. Save the final engineered masterpiece
df.to_csv(file_path, index=False)

print("---")
print(f"Features created: {', '.join(lc_mapping.values())}")
print("Checkpoint: Landcover encoded. Phase 3 Feature Engineering is officially COMPLETE.")

One-Hot Encoding Landcover Features...
---
Features created: lc_forest, lc_grassland, lc_crop, lc_urban, lc_wetland
Checkpoint: Landcover encoded. Phase 3 Feature Engineering is officially COMPLETE.


In [31]:
print(df.columns.tolist())

['B11', 'B12', 'B2', 'B3', 'B4', 'B8', 'bd_0-5', 'bd_15-30', 'bd_5-15', 'bsi', 'cec_0-5', 'cec_15-30', 'cec_5-15', 'clay_0-5', 'clay_15-30', 'clay_5-15', 'dem', 'evi', 'ndmi', 'ndvi', 'pet_apr', 'pet_aug', 'pet_dec', 'pet_feb', 'pet_jan', 'pet_jul', 'pet_jun', 'pet_mar', 'pet_may', 'pet_nov', 'pet_oct', 'pet_sep', 'ph_0-5', 'ph_15-30', 'ph_5-15', 'rain_apr', 'rain_aug', 'rain_dec', 'rain_feb', 'rain_jan', 'rain_jul', 'rain_jun', 'rain_mar', 'rain_may', 'rain_nov', 'rain_oct', 'rain_sep', 'sand_0-5', 'sand_15-30', 'sand_5-15', 'savi', 'silt_0-5', 'silt_15-30', 'silt_5-15', 'slope', 'soc_0-5', 'soc_15-30', 'soc_5-15', 'worldcover', 'clay_0_30', 'sand_0_30', 'silt_0_30', 'soc_0_30', 'bd_0_30', 'clay_sand_ratio', 'silt_clay_ratio', 'sand_silt_ratio', 'som', 'porosity', 'ndwi', 'gndvi', 'brightness_index', 'ndti', 'pool_IOM', 'pool_DPM', 'pool_RPM', 'pool_BIO', 'pool_HUM', 'annual_rain', 'mean_temp', 'temp_std', 'rain_cv', 'aridity_index', 'aspect_sin', 'aspect_cos', 'twi']


In [34]:
import pandas as pd
import numpy as np
import rasterio

# 1. Define Paths
# Use any of your clipped/standardized .tif files as a reference for the grid
ref_raster_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clipped\dem.tif"
csv_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"

# 2. Extract Coordinates from the Raster Grid
with rasterio.open(ref_raster_path) as src:
    # Get the coordinate arrays for the entire grid
    # This creates two 2D arrays of the same shape as the raster
    # containing the X and Y UTM coordinates for every pixel center.
    cols, rows = np.meshgrid(np.arange(src.width), np.arange(src.height))
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    
    # Flatten them to 1D
    x_coords = np.array(xs).flatten()
    y_coords = np.array(ys).flatten()
    
    # Create a mask for the NoData values (identical to how you built the CSV)
    data = src.read(1)
    mask = (data != src.nodata) & (~np.isnan(data))
    mask_flat = mask.flatten()

# 3. Filter coordinates to match the CSV rows
# The CSV only contains valid "inside-farm" pixels. 
# We filter the grid coordinates using the same mask.
valid_x = x_coords[mask_flat]
valid_y = y_coords[mask_flat]

# 4. Load the CSV and append the coordinates
df = pd.read_csv(csv_path)

# Safety Check: The number of valid pixels must match the CSV row count
if len(df) == len(valid_x):
    df['x'] = valid_x
    df['y'] = valid_y
    df.to_csv(csv_path, index=False)
    print("Success: 'x' and 'y' coordinate columns added to your dataset.")
else:
    print(f"Error: Row mismatch! CSV has {len(df)} rows, but raster mask has {len(valid_x)} pixels.")

Success: 'x' and 'y' coordinate columns added to your dataset.


In [35]:
import pandas as pd
import numpy as np

# 1. Load the engineered dataset
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

print("Applying Spatial Positional Encoding...")

# 2. Coordinate Normalization (Min-Max Scaling)
# This maps your UTM coordinates to a 0-1 range for GNN stability
df['x_norm'] = (df['x'] - df['x'].min()) / (df['x'].max() - df['x'].min())
df['y_norm'] = (df['y'] - df['y'].min()) / (df['y'].max() - df['y'].min())

# 3. Fourier Positional Encoding
# We create 3 levels of frequency (k=1, 2, 3) for both X and Y
for k in [1, 2, 3]:
    # Frequency scaling: 2^k * PI
    freq = (2**k) * np.pi
    
    # Generate Sine and Cosine pairs
    df[f'pe_x_sin_{k}'] = np.sin(freq * df['x_norm'])
    df[f'pe_x_cos_{k}'] = np.cos(freq * df['x_norm'])
    df[f'pe_y_sin_{k}'] = np.sin(freq * df['y_norm'])
    df[f'pe_y_cos_{k}'] = np.cos(freq * df['y_norm'])

# 4. Save the spatially enhanced dataset
df.to_csv(file_path, index=False)

print("---")
print("Spatial Features Created: x_norm, y_norm, and 12 Fourier Encoding columns.")
print("Checkpoint: Positional encoding enhancement complete. Ready for GNN architecture.")

Applying Spatial Positional Encoding...
---
Spatial Features Created: x_norm, y_norm, and 12 Fourier Encoding columns.
Checkpoint: Positional encoding enhancement complete. Ready for GNN architecture.


In [36]:
import pandas as pd

# 1. Load the dataset
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

print("Engineering Cross-Domain Interaction Features...")

# 2. Clay x Rainfall (Stabilization Proxy)
# High clay + High rain usually = High potential for stable carbon pools
df['clay_rain'] = df['clay_0_30'] * df['annual_rain']

# 3. SOC x NDVI (Carbon Input Proxy)
# Existing SOC paired with high vegetation vigor indicates an 'active' carbon pump
# Using ndvi_mean (if available) or the baseline ndvi
ndvi_col = 'ndvi_mean' if 'ndvi_mean' in df.columns else 'ndvi'
df['soc_ndvi'] = df['soc_0_30'] * df[ndvi_col]

# 4. Slope x Rainfall (Leaching/Erosion Proxy)
# Steeper slopes with heavy rain suggest areas where carbon might be lost to runoff
df['slope_rain'] = df['slope'] * df['annual_rain']

# 5. Temperature x Moisture (Decomposition Rate Proxy)
# High temp + High ndmi (moisture) = Rapid microbial decomposition
df['temp_moisture'] = df['mean_temp'] * df['ndmi']

# 6. Save Updated Dataset
df.to_csv(file_path, index=False)

print("---")
print("Features created: clay_rain, soc_ndvi, slope_rain, temp_moisture")
print("Checkpoint: Interaction features finalized.")

Engineering Cross-Domain Interaction Features...
---
Features created: clay_rain, soc_ndvi, slope_rain, temp_moisture
Checkpoint: Interaction features finalized.


In [42]:
import os
import glob
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

input_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\climate"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\aligned_temp"
reference = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\aligned\B2.tif"

os.makedirs(output_dir, exist_ok=True)

with rasterio.open(reference) as ref:
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width = ref.width
    ref_height = ref.height
    ref_profile = ref.profile.copy()
    
temp_files = glob.glob(os.path.join(input_dir, "*temp*.tif"))

for filepath in temp_files:
    filename = os.path.basename(filepath)
    out_path = os.path.join(output_dir, filename)

    with rasterio.open(filepath) as src:
        destination = np.empty(
            (ref_height, ref_width),
            dtype=src.dtypes[0]
        )

        reproject(
            source=rasterio.band(src, 1),
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear
        )

        profile = ref_profile.copy()
        profile.update(dtype=src.dtypes[0], count=1)

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(destination, 1)

print("Temperature rasters aligned.")

Temperature rasters aligned.


In [46]:
import pandas as pd
import glob
import os
import rasterio
import numpy as np

csv_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
aligned_temp_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\aligned_temp"
reference = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\aligned\B2.tif"

df = pd.read_csv(csv_path)

with rasterio.open(reference) as ref:
    ref_arr = ref.read(1)
    ref_nodata = ref.nodata

    ref_mask = (ref_arr != ref_nodata) & (~np.isnan(ref_arr))

temp_files = glob.glob(os.path.join(aligned_temp_dir, "*temp*.tif"))

for f in temp_files:
    col_name = os.path.basename(f).replace(".tif", "")

    with rasterio.open(f) as src:
        arr = src.read(1)

        values = arr[ref_mask]

        if len(values) != len(df):
            raise ValueError(
                f"{col_name}: {len(values)} != {len(df)}"
            )

        df[col_name] = values

df.to_csv(csv_path, index=False)

print(f"Merged {len(temp_files)} temperature columns.")

Merged 12 temperature columns.


In [49]:
import os
import glob
import rasterio
import pandas as pd
import numpy as np

# 1. Paths - Update these to where your RAW temperature files are
raw_temp_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\climate"
csv_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
output_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\standardized"

df = pd.read_csv(csv_path)

# 2. Find all temperature files (looking for tavg or temp)
temp_files = glob.glob(os.path.join(raw_temp_dir, "*tavg*.tif")) + glob.glob(os.path.join(raw_temp_dir, "*temp*.tif"))

print(f"Found {len(temp_files)} raw temperature files. Processing...")

for f in temp_files:
    filename = os.path.basename(f)
    # Standardize name to temp_mon.tif
    month = next((m for m in ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'] if m in filename.lower()), "mean")
    col_name = f"temp_{month}"
    
    with rasterio.open(f) as src:
        data = src.read(1).astype('float32')
        nodata = src.nodata
        
        # Unit Correction Logic: 
        # If mean > 100, it's Kelvin. If mean is very small, it might be scaled C (x10).
        valid_mask_raster = (data != nodata) & (~np.isnan(data))
        mean_val = np.mean(data[valid_mask_raster])
        
        if mean_val > 200: # Kelvin
            data[valid_mask_raster] -= 273.15
        elif mean_val > 50: # Likely scaled Celsius (e.g. 250 for 25.0C)
            data[valid_mask_raster] /= 10.0
            
        # 3. Align with CSV mask
        # We need to ensure we only grab the pixels that match the existing CSV rows
        # This assumes your CSV and these rasters are now in the same CRS/Resolution
        flat_data = data.flatten()
        # You might need to re-apply the AOI mask here if row counts differ
        # For now, we'll assign the valid pixels directly if counts match
        if len(data[valid_mask_raster]) == len(df):
            df[col_name] = data[valid_mask_raster]
            print(f"Successfully added {col_name} with real values.")
        else:
            print(f"Skipping {col_name}: Raster valid pixels ({len(data[valid_mask_raster])}) != CSV rows ({len(df)})")

# 4. Save the repaired CSV
df.to_csv(csv_path, index=False)
print("\nCSV Repaired. Check your temp_ columns now—they should be ~20-35, not 0.")

Found 12 raw temperature files. Processing...
Skipping temp_apr: Raster valid pixels (15) != CSV rows (143)
Skipping temp_aug: Raster valid pixels (15) != CSV rows (143)
Skipping temp_dec: Raster valid pixels (15) != CSV rows (143)
Skipping temp_feb: Raster valid pixels (15) != CSV rows (143)
Skipping temp_jan: Raster valid pixels (15) != CSV rows (143)
Skipping temp_jul: Raster valid pixels (15) != CSV rows (143)
Skipping temp_jun: Raster valid pixels (15) != CSV rows (143)
Skipping temp_mar: Raster valid pixels (15) != CSV rows (143)
Skipping temp_may: Raster valid pixels (15) != CSV rows (143)
Skipping temp_nov: Raster valid pixels (15) != CSV rows (143)
Skipping temp_oct: Raster valid pixels (15) != CSV rows (143)
Skipping temp_sep: Raster valid pixels (15) != CSV rows (143)

CSV Repaired. Check your temp_ columns now—they should be ~20-35, not 0.


In [51]:
import pandas as pd
import numpy as np
import rasterio
from pyproj import Transformer
import os
import glob

# 1. Paths
csv_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
raw_temp_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\climate"

df = pd.read_csv(csv_path)

# 2. Setup Coordinate Transformer
# Source: UTM Zone 43N (EPSG:32643) -> Target: WGS84 Lat/Lon (EPSG:4326)
transformer = Transformer.from_crs("EPSG:32643", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(df['x'].values, df['y'].values)

# 3. Find Temp Files
temp_files = glob.glob(os.path.join(raw_temp_dir, "*tavg*.tif")) + glob.glob(os.path.join(raw_temp_dir, "*temp*.tif"))

print(f"Sampling {len(temp_files)} climate rasters with coordinate transformation...")

for f in temp_files:
    filename = os.path.basename(f).lower()
    month = next((m for m in ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'] if m in filename), "mean")
    col_name = f"temp_{month}"
    
    with rasterio.open(f) as src:
        # Sample using the transformed Lat/Lon
        coords = list(zip(lon, lat))
        samples = [val[0] for val in src.sample(coords)]
        temp_vals = np.array(samples, dtype=float)
        
        # Handle NoData and Scaling
        nodata = src.nodata
        valid_mask = (temp_vals != nodata) & (temp_vals > -100) & (~np.isnan(temp_vals))
        
        if not any(valid_mask):
            print(f"!! Still finding 0 valid points for {col_name}. Check if Lat/Lon matches raster bounds.")
            continue

        mean_check = np.mean(temp_vals[valid_mask])
        
        # Unit Correction (Kelvin -> C or Scaled C -> C)
        if mean_check > 200: 
            temp_vals[valid_mask] -= 273.15
        elif mean_check > 50 or mean_check < -10: # Likely scaled by 10 (e.g. 250 for 25.0C)
            temp_vals[valid_mask] /= 10.0
            
        df[col_name] = temp_vals
        print(f"Updated {col_name}. Mean: {np.mean(temp_vals[valid_mask]):.2f} C")

# 4. Final Save
df.to_csv(csv_path, index=False)
print("\nTransformation complete. Please check the CSV for non-zero values.")

Sampling 12 climate rasters with coordinate transformation...
Updated temp_apr. Mean: 27.74 C
Updated temp_aug. Mean: 23.79 C
Updated temp_dec. Mean: 20.71 C
Updated temp_feb. Mean: 24.52 C
Updated temp_jan. Mean: 21.86 C
Updated temp_jul. Mean: 23.54 C
Updated temp_jun. Mean: 24.77 C
Updated temp_mar. Mean: 27.22 C
Updated temp_may. Mean: 26.12 C
Updated temp_nov. Mean: 22.62 C
Updated temp_oct. Mean: 23.98 C
Updated temp_sep. Mean: 23.97 C

Transformation complete. Please check the CSV for non-zero values.


In [53]:
import pandas as pd
import numpy as np
import rasterio
from pyproj import Transformer
import os
import glob

# 1. Paths
csv_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
raw_climate_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\climate"
output_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\rothc_input.csv"

df = pd.read_csv(csv_path)

# 2. Coordinate Transformation (UTM 43N -> WGS84 Lat/Lon)
# This prevents the "Zero Value" error caused by CRS mismatch
transformer = Transformer.from_crs("EPSG:32643", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(df['x'].values, df['y'].values)
coords = list(zip(lon, lat))

# 3. Initialize RothC Table
rothc_df = df[['x', 'y', 'clay_0_30']].copy()
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

def sample_raster(pattern, month):
    files = glob.glob(os.path.join(raw_climate_dir, f"*{pattern}*{month}*.tif"))
    if not files: return np.zeros(len(df))
    
    with rasterio.open(files[0]) as src:
        # Sample using transformed coordinates
        vals = np.array([v[0] for v in src.sample(coords)], dtype=float)
        # Clean NoData and apply common scaling (e.g., WorldClim temp is x10)
        vals[vals == src.nodata] = np.nan
        if pattern == 'temp' or 'tavg' in pattern:
            if np.nanmean(vals) > 200: vals -= 273.15 # Kelvin to C
            elif np.nanmean(vals) > 50: vals /= 10.0  # Scale factor 10
        return np.nan_to_num(vals)

print("Building monthly RothC parameters...")
for m in months:
    # A. Climate Drivers
    rothc_df[f'temp_{m}'] = sample_raster('tavg', m)
    rothc_df[f'rain_{m}'] = sample_raster('prec', m)
    rothc_df[f'pet_{m}'] = sample_raster('pet', m)
    
    # B. Soil Cover (1=Protected, 0=Bare)
    # Using your worldcover labels (e.g., 2 for Grassland)
    rothc_df[f'cover_{m}'] = df['worldcover'].apply(lambda x: 1 if x in [1, 2] else 0)
    
    # C. Carbon Input (Cin) - Proxy from NDVI
    # We estimate C-input is proportional to vegetation vigor
    ndvi_col = 'ndvi_mean' if 'ndvi_mean' in df.columns else 'ndvi'
    rothc_df[f'cin_{m}'] = df[ndvi_col] * 0.5 

# 4. Final Save
rothc_df.to_csv(output_path, index=False)
print(f"Success! RothC input table ready at: {output_path}")

Building monthly RothC parameters...
Success! RothC input table ready at: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\rothc_input.csv


In [55]:
import pandas as pd
import numpy as np
import rasterio
from pyproj import Transformer
import os
import glob

# 1. Paths
csv_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
raw_climate_dir = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\raw\climate"
output_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\rothc_input.csv"

df = pd.read_csv(csv_path)
transformer = Transformer.from_crs("EPSG:32643", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(df['x'].values, df['y'].values)
coords = list(zip(lon, lat))

rothc_df = df[['x', 'y', 'clay_0_30']].copy()
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

def sample_variable(var_keywords, month):
    # Search for ANY file that contains the month and one of the keywords
    found_files = []
    for kw in var_keywords:
        pattern = os.path.join(raw_climate_dir, f"*{kw}*{month}*.tif")
        found_files.extend(glob.glob(pattern))
    
    if not found_files:
        return None, None
    
    target_file = found_files[0]
    with rasterio.open(target_file) as src:
        vals = np.array([v[0] for v in src.sample(coords)], dtype=float)
        # Handle NoData
        vals[vals == src.nodata] = np.nan
        return vals, target_file

print("--- Starting RothC Input Build ---")
for m in months:
    # A. Temperature (Try tavg, temp, or tmean)
    t_vals, t_file = sample_variable(['tavg', 'temp', 'tmean'], m)
    if t_vals is not None:
        # Auto-Correction for Kelvin/Scaled Celsius
        m_val = np.nanmean(t_vals)
        if m_val > 200: t_vals -= 273.15
        elif m_val > 50 or m_val < -10: t_vals /= 10.0
        rothc_df[f'temp_{m}'] = np.nan_to_num(t_vals)
        print(f"[{m}] Temp: Found {os.path.basename(t_file)} (Mean: {np.nanmean(t_vals):.2f})")
    else:
        print(f"[{m}] Temp: NO FILE FOUND")

    # B. Rainfall (Try prec, rain, or precip)
    r_vals, r_file = sample_variable(['prec', 'rain', 'precip'], m)
    if r_vals is not None:
        rothc_df[f'rain_{m}'] = np.nan_to_num(r_vals)
        print(f"[{m}] Rain: Found {os.path.basename(r_file)}")
    else:
        print(f"[{m}] Rain: NO FILE FOUND")

    # C. PET (Try pet, et0)
    p_vals, p_file = sample_variable(['pet', 'et0'], m)
    if p_vals is not None:
        rothc_df[f'pet_{m}'] = np.nan_to_num(p_vals)
        print(f"[{m}] PET:  Found {os.path.basename(p_file)}")
    else:
        print(f"[{m}] PET:  NO FILE FOUND")

    # D. Derived Features
    rothc_df[f'cover_{m}'] = df['worldcover'].apply(lambda x: 1 if x in [1, 2] else 0)
    ndvi_col = 'ndvi_mean' if 'ndvi_mean' in df.columns else 'ndvi'
    rothc_df[f'cin_{m}'] = df[ndvi_col] * 0.5

# 5. Final Save
rothc_df.to_csv(output_path, index=False)
print("--- Build Finished ---")

--- Starting RothC Input Build ---
[jan] Temp: Found temp_jan.tif (Mean: 21.86)
[jan] Rain: Found rain_jan.tif
[jan] PET:  Found pet_jan.tif
[feb] Temp: Found temp_feb.tif (Mean: 24.52)
[feb] Rain: Found rain_feb.tif
[feb] PET:  Found pet_feb.tif
[mar] Temp: Found temp_mar.tif (Mean: 27.22)
[mar] Rain: Found rain_mar.tif
[mar] PET:  Found pet_mar.tif
[apr] Temp: Found temp_apr.tif (Mean: 27.74)
[apr] Rain: Found rain_apr.tif
[apr] PET:  Found pet_apr.tif
[may] Temp: Found temp_may.tif (Mean: 26.12)
[may] Rain: Found rain_may.tif
[may] PET:  Found pet_may.tif
[jun] Temp: Found temp_jun.tif (Mean: 24.77)
[jun] Rain: Found rain_jun.tif
[jun] PET:  Found pet_jun.tif
[jul] Temp: Found temp_jul.tif (Mean: 23.54)
[jul] Rain: Found rain_jul.tif
[jul] PET:  Found pet_jul.tif
[aug] Temp: Found temp_aug.tif (Mean: 23.79)
[aug] Rain: Found rain_aug.tif
[aug] PET:  Found pet_aug.tif
[sep] Temp: Found temp_sep.tif (Mean: 23.97)
[sep] Rain: Found rain_sep.tif
[sep] PET:  Found pet_sep.tif
[oct] Temp:

In [60]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# =========================================================
# 1. Load dataset
# =========================================================
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"

df = pd.read_csv(file_path)

print("Dataset loaded.")

# =========================================================
# 2. Define target and features
# =========================================================
target = 'soc_0_30'

drop_cols = [
    target,
    'x',
    'y',
    'worldcover'
]

drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = df[target]

print(f"Initial feature count: {X.shape[1]}")

# =========================================================
# 3. Remove completely empty columns
# =========================================================
all_nan_cols = X.columns[X.isna().all()].tolist()

if len(all_nan_cols) > 0:
    print("\nDropping all-NaN columns:")
    print(all_nan_cols)

    X = X.drop(columns=all_nan_cols)

# =========================================================
# 4. Remove constant columns
# =========================================================
constant_cols = X.columns[X.nunique(dropna=True) <= 1].tolist()

if len(constant_cols) > 0:
    print("\nDropping constant columns:")
    print(constant_cols)

    X = X.drop(columns=constant_cols)

print(f"\nFeature count after cleanup: {X.shape[1]}")

# =========================================================
# 5. Impute partial missing values
# =========================================================
imputer = SimpleImputer(strategy='median')

X_imputed = imputer.fit_transform(X)

X_imputed_df = pd.DataFrame(
    X_imputed,
    columns=X.columns
)

print("\nMissing values imputed.")

# =========================================================
# 6. Scale for LASSO
# =========================================================
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_imputed_df)

print("Features standardized for LASSO.")

print(f"\nStarting feature selection from {X_imputed_df.shape[1]} features...")

# =========================================================
# 7. Random Forest Importance
# =========================================================
print("\nRunning Random Forest importance...")

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_imputed_df, y)

rf_importance = pd.Series(
    rf.feature_importances_,
    index=X_imputed_df.columns
)

# =========================================================
# 8. LASSO Importance
# =========================================================
print("Running LASSO importance...")

lasso = LassoCV(
    cv=5,
    random_state=42,
    max_iter=10000
)

lasso.fit(X_scaled, y)

lasso_importance = pd.Series(
    np.abs(lasso.coef_),
    index=X_imputed_df.columns
)

# =========================================================
# 9. Mutual Information Importance
# =========================================================
print("Running Mutual Information importance...")

mi_scores = mutual_info_regression(
    X_imputed_df,
    y,
    random_state=42
)

mi_importance = pd.Series(
    mi_scores,
    index=X_imputed_df.columns
)

# =========================================================
# 10. Rank Fusion
# =========================================================
print("Combining rankings...")

rank_df = pd.DataFrame({
    'RF_rank': rf_importance.rank(ascending=False),
    'LASSO_rank': lasso_importance.rank(ascending=False),
    'MI_rank': mi_importance.rank(ascending=False)
})

rank_df['mean_rank'] = rank_df.mean(axis=1)

rank_df = rank_df.sort_values('mean_rank')

# Top K features
TOP_K = 25

selected_features = rank_df.head(TOP_K).index.tolist()

# =========================================================
# 11. Save selected feature metadata
# =========================================================
output_meta = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\selected_features.csv"

pd.DataFrame({
    'feature': selected_features
}).to_csv(output_meta, index=False)

# =========================================================
# 12. Create final model-ready dataset
# =========================================================
final_cols = selected_features + [target]

if 'x' in df.columns:
    final_cols.append('x')

if 'y' in df.columns:
    final_cols.append('y')

df_selected = df[final_cols]

output_final = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\final_model_input.csv"

df_selected.to_csv(output_final, index=False)

# =========================================================
# 13. Summary
# =========================================================
print("\n===================================")
print("Feature Selection Complete")
print("===================================")
print(f"Selected {len(selected_features)} features")
print("\nTop 10 Selected Features:")
print(selected_features[:10])

print(f"\nSaved feature list -> {output_meta}")
print(f"Saved final dataset -> {output_final}")

Dataset loaded.
Initial feature count: 119

Dropping all-NaN columns:
['mean_temp', 'temp_std', 'temp_moisture']

Dropping constant columns:
['rain_feb', 'rain_jan', 'lc_forest', 'lc_grassland', 'lc_crop', 'lc_urban', 'lc_wetland']

Feature count after cleanup: 109

Missing values imputed.
Features standardized for LASSO.

Starting feature selection from 109 features...

Running Random Forest importance...
Running LASSO importance...
Running Mutual Information importance...
Combining rankings...

Feature Selection Complete
Selected 25 features

Top 10 Selected Features:
['soc_15-30', 'soc_0-5', 'soc_5-15', 'som', 'clay_5-15', 'clay_15-30', 'bd_0-5', 'bd_0_30', 'cec_5-15', 'bd_15-30']

Saved feature list -> C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\selected_features.csv
Saved final dataset -> C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\final_model_input.csv


In [68]:
import os
import pandas as pd
import numpy as np

# 1. Load the merged feature set
file_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\clean_master_engineered.csv"
df = pd.read_csv(file_path)

print(f"Preparing RothC input table for {len(df)} sites...")

# 2. Define constants and monthly lists
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
rows = []

# 3. Iterate through every pixel (site)
for idx, pixel in df.iterrows():
    site_id = f"site_{idx:04d}" # Unique ID for each pixel
    
    for i, month in enumerate(months):
        month_lower = month.lower()
        
        # Extract variables using your column naming convention
        # Note: 'clay_0_30' is constant for all 12 months for a single site
        site_data = {
            'site_id': site_id,
            'x': pixel['x'],
            'y': pixel['y'],
            'month': month,
            'temperature': pixel.get(f'temp_{month_lower}', 0),
            'rainfall': pixel.get(f'rain_{month_lower}', 0),
            'pet': pixel.get(f'pet_{month_lower}', 0),
            'clay': pixel['clay_0_30'],
            # Carbon Input Proxy: Normalized NDVI/EVI scaled to a baseline
            'carbon_input': pixel.get('ndvi_mean', 0.5) * 0.8, 
            # Soil Cover: Mapping WorldCover to RothC status
            'soil_cover': 'covered' if pixel['worldcover'] in [1, 2] else 'bare'
        }
        rows.append(site_data)

# 4. Create the Long-Format DataFrame
rothc_long_df = pd.DataFrame(rows)

# 5. Final Save
output_path = r"C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\features\rothc_input_long.csv"
rothc_long_df.to_csv(output_path, index=False)

print("---")
print(f"RothC Input Table Built: {output_path}")
print(f"Total entries: {len(rothc_long_df)} ({len(df)} sites x 12 months)")

Preparing RothC input table for 143 sites...
---
RothC Input Table Built: C:\Users\AYUSH SINGH\Documents\GitHub\AgroGraphAI\data\processed\features\rothc_input_long.csv
Total entries: 1716 (143 sites x 12 months)
